In [ ]:
import pandas as pd
print("Pandas успешно импортирован, версия:", pd.__version__)

In [ ]:
import pandas as pd

# Загружаем датасет
df = pd.read_csv('Space_Corrected.csv')

# Посмотрим на первые 5 строк и общую информацию
print("Первые 5 строк:")
print(df.head())

print("\nИнформация о колонках (типы данных и пропуски):")
print(df.info())

In [ ]:
import pandas as pd

# Загружаем данные (заново для чистоты)
df = pd.read_csv('Space_Corrected.csv')

# 1. Удаляем дублирующиеся индексы
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'])

# 2. Преобразуем 'Datum' в формат datetime
df['Datum'] = pd.to_datetime(df['Datum'], errors='coerce')

# 3. Посмотрим на распределение статусов миссий
print("Распределение статусов миссий:")
print(df['Status Mission'].value_counts())

print("\nПервые 5 строк после очистки:")
print(df.head())

In [ ]:
# Установим Datum как индекс (это сделает данные временным рядом)
df = df.set_index('Datum')

# Посмотрим на количество запусков по годам
print("Количество запусков по годам:")
print(df.resample('Y').size())

# Визуализируем количество запусков по годам
import matplotlib.pyplot as plt

df.resample('Y').size().plot(kind='bar', figsize=(12, 6), color='skyblue')
plt.title('Количество космических запусков по годам')
plt.xlabel('Год')
plt.ylabel('Количество запусков')
plt.grid(axis='y')
plt.show()

In [ ]:
# Сгруппируем по годам и посчитаем количество успехов и неудач
status_by_year = df.groupby(df.index.year)['Status Mission'].value_counts().unstack().fillna(0)

# Выведем первые несколько строк
print(status_by_year.head())

# Построим график
status_by_year.plot(kind='bar', stacked=True, figsize=(14, 6), colormap='viridis')
plt.title('Успешные и неудачные запуски по годам')
plt.xlabel('Год')
plt.ylabel('Количество запусков')
plt.legend(title='Статус миссии')
plt.grid(axis='y')
plt.show()

In [ ]:
# Извлекаем месяц из даты
df['Month'] = df.index.month

# Считаем количество запусков по месяцам за все годы
monthly_counts = df['Month'].value_counts().sort_index()

# Строим график
plt.figure(figsize=(10, 5))
monthly_counts.plot(kind='bar', color='coral')
plt.title('Количество запусков по месяцам (все годы)')
plt.xlabel('Месяц')
plt.ylabel('Количество запусков')
plt.xticks(range(1, 13), ['Янв', 'Фев', 'Мар', 'Апр', 'Май', 'Июн', 'Июл', 'Авг', 'Сен', 'Окт', 'Ноя', 'Дек'])
plt.grid(axis='y')
plt.show()

In [ ]:
# 1. Создаём колонку 'IsSuccess' (1 - успех, 0 - всё остальное)
df['IsSuccess'] = (df['Status Mission'] == 'Success').astype(int)

# 2. Группируем по месяцам и считаем долю успешных запусков
monthly_success = df.resample('M')['IsSuccess'].mean() * 100  # в процентах

# 3. Убираем пустые значения (если в какой-то месяц не было запусков)
monthly_success = monthly_success.dropna()

# 4. Смотрим на первые 10 значений
print(monthly_success.head(10))

# 5. Строим график
plt.figure(figsize=(12, 6))
monthly_success.plot(color='green')
plt.title('Процент успешных запусков по месяцам')
plt.xlabel('Дата')
plt.ylabel('Успешность (%)')
plt.grid(True)
plt.show()

In [ ]:
!pip install statsforecast

In [ ]:
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, ETS, Theta

# 1. Создаём датафрейм в формате, ожидаемом statsforecast
# Нам нужны колонки: 'ds' (дата), 'y' (значение), 'unique_id' (идентификатор ряда)
ts_data = monthly_success.reset_index()
ts_data.columns = ['ds', 'y']
ts_data['unique_id'] = 'success_rate'

# 2. Смотрим на итоговый формат
print(ts_data.head())
print(f"\nКоличество точек данных: {len(ts_data)}")

In [ ]:
import pandas as pd
import numpy as np
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, Theta

# 1. Подготавливаем данные в нужном формате (если monthly_success уже есть в памяти)
ts_data = monthly_success.reset_index()
ts_data.columns = ['ds', 'y']
ts_data['unique_id'] = 'success_rate'

# 2. Создаём список моделей с исправленным названием
models = [
    AutoARIMA(season_length=12),       # 12 месяцев — сезонность
    AutoETS(season_length=12),         # Автоматический ETS (исправлено!)
    Theta(season_length=12)            # Метод Теты
]

# 3. Инициализируем StatsForecast
sf = StatsForecast(
    models=models,
    freq='M',            # Ежемесячные данные
    n_jobs=-1            # Использовать все ядра процессора
)

# 4. Смотрим, что получилось
print("Модели успешно инициализированы!")
print(f"Данных: {len(ts_data)} точек")
print(ts_data.head())

In [ ]:
# Делаем прогноз на 12 месяцев вперёд
forecast = sf.forecast(h=12, df=ts_data)

# Выводим прогноз
print("Прогноз на 12 месяцев вперёд:")
print(forecast)

# Строим график
import matplotlib.pyplot as plt

# Создаём фигуру
plt.figure(figsize=(14, 7))

# Строим исторические данные
plt.plot(ts_data['ds'], ts_data['y'], label='Фактические данные', color='black', alpha=0.5)

# Строим прогноз для каждой модели
models_list = ['AutoARIMA', 'AutoETS', 'Theta']
colors = ['blue', 'red', 'green']

for model, color in zip(models_list, colors):
    # Фильтруем прогноз для текущей модели
    model_forecast = forecast[forecast['model'] == model]
    plt.plot(model_forecast['ds'], model_forecast['mean'], label=model, color=color, linewidth=2)

plt.title('Прогноз процента успешных запусков на 2021 год (3 модели)')
plt.xlabel('Дата')
plt.ylabel('Успешность (%)')
plt.legend()
plt.grid(True)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, Theta

# =============================================
# 1. ПОДГОТОВКА ДАННЫХ
# =============================================
# Предполагается, что переменная monthly_success уже существует из предыдущего шага.
# Если вы перезапускали ядро, раскомментируйте строки ниже и загрузите данные заново.
# df = pd.read_csv('Space_Corrected.csv')
# df = df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'])
# df['Datum'] = pd.to_datetime(df['Datum'], errors='coerce')
# df = df.set_index('Datum')
# df['IsSuccess'] = (df['Status Mission'] == 'Success').astype(int)
# monthly_success = df.resample('M')['IsSuccess'].mean() * 100
# monthly_success = monthly_success.dropna()

# Форматируем данные для библиотеки statsforecast
ts_data = monthly_success.reset_index()
ts_data.columns = ['ds', 'y']
ts_data['unique_id'] = 'success_rate'

print(f"Данных загружено: {len(ts_data)} точек")
print(ts_data.head())

# =============================================
# 2. ИНИЦИАЛИЗАЦИЯ МОДЕЛЕЙ
# =============================================
models = [
    AutoARIMA(season_length=12),
    AutoETS(season_length=12),
    Theta(season_length=12)
]

sf = StatsForecast(
    models=models,
    freq='M',
    n_jobs=-1
)

print("\nМодели успешно инициализированы!")

# =============================================
# 3. ОБУЧЕНИЕ И ПРОГНОЗ НА 12 МЕСЯЦЕВ
# =============================================
forecast = sf.forecast(h=12, df=ts_data)

print("\nПрогноз на 12 месяцев вперёд:")
print(forecast)

# =============================================
# 4. ВИЗУАЛИЗАЦИЯ РЕЗУЛЬТАТОВ
# =============================================
plt.figure(figsize=(14, 7))

# Исторические данные
plt.plot(ts_data['ds'], ts_data['y'], label='Фактические данные', color='black', alpha=0.5, linewidth=1)

# Прогнозы каждой модели
models_list = ['AutoARIMA', 'AutoETS', 'Theta']
colors = ['blue', 'red', 'green']

for model, color in zip(models_list, colors):
    plt.plot(forecast['ds'], forecast[model], label=model, color=color, linewidth=2)

plt.title('Прогноз процента успешных запусков на 2021 год (сравнение 3 моделей)')
plt.xlabel('Дата')
plt.ylabel('Успешность (%)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

print("\n✅ Скрипт выполнен успешно!")

In [ ]:
!pip install mlforecast neuralforecast

In [ ]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Подготовка данных (используем ts_data из предыдущего шага)
# Добавляем лаги (признаки из прошлых значений)
mlf = MLForecast(
    models=[
        LGBMRegressor(random_state=42, verbose=-1),
        XGBRegressor(random_state=42, verbosity=0),
        CatBoostRegressor(random_state=42, verbose=0)
    ],
    freq='M',
    lags=[1, 2, 3, 6, 12],  # Лаги: 1,2,3,6,12 месяцев назад
    target_transforms=[Differences([1])]  # Берём разность первого порядка
)

# Обучаем модели
mlf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')

# Делаем прогноз на 12 месяцев
ml_forecast = mlf.predict(h=12)

print("ML-прогноз готов:")
print(ml_forecast.head())

In [ ]:
!pip install catboost

In [ ]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Подготовка данных (используем ts_data из предыдущего шага)
# Добавляем лаги (признаки из прошлых значений)
mlf = MLForecast(
    models=[
        LGBMRegressor(random_state=42, verbose=-1),
        XGBRegressor(random_state=42, verbosity=0),
        CatBoostRegressor(random_state=42, verbose=0)
    ],
    freq='M',
    lags=[1, 2, 3, 6, 12],  # Лаги: 1,2,3,6,12 месяцев назад
    target_transforms=[Differences([1])]  # Берём разность первого порядка
)

# Обучаем модели
mlf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')

# Делаем прогноз на 12 месяцев
ml_forecast = mlf.predict(h=12)

print("ML-прогноз готов:")
print(ml_forecast.head())

In [ ]:
# 1. Создаём колонку 'IsSuccess' (1 - успех, 0 - всё остальное)
df['IsSuccess'] = (df['Status Mission'] == 'Success').astype(int)

# 2. Группируем по месяцам и считаем долю успешных запусков
monthly_success = df.resample('M')['IsSuccess'].mean() * 100  # в процентах

# 3. ВАЖНО: Заполняем пропущенные месяцы нулями (если месяц пропущен, значит не было запусков)
monthly_success = monthly_success.fillna(0)

# 4. Снова создаём ts_data для mlforecast (с исправленным рядом)
ts_data = monthly_success.reset_index()
ts_data.columns = ['ds', 'y']
ts_data['unique_id'] = 'success_rate'

# 5. Проверяем, что пропусков больше нет
print(f"Количество точек: {len(ts_data)}")
print(ts_data.head(10))

In [ ]:
ML-прогноз готов:
       unique_id         ds  LightGBM  XGBoost  CatBoost
0   success_rate 2021-09-30   ...       ...       ...

In [ ]:
import pandas as pd
import numpy as np
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor

# Подготовка данных (используем ts_data из предыдущего шага)
# Добавляем лаги (признаки из прошлых значений)
mlf = MLForecast(
    models=[
        LGBMRegressor(random_state=42, verbose=-1),
        XGBRegressor(random_state=42, verbosity=0),
        CatBoostRegressor(random_state=42, verbose=0)
    ],
    freq='M',
    lags=[1, 2, 3, 6, 12],
    target_transforms=[Differences([1])]
)

# Обучаем модели
mlf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')

# Делаем прогноз на 12 месяцев
ml_forecast = mlf.predict(h=12)

print("ML-прогноз готов:")
print(ml_forecast.head(12))

In [ ]:
# Установка библиотеки для DL (если ещё не установлена)
!pip install neuralforecast

import pandas as pd
import numpy as np
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, TFT
from neuralforecast.losses.pytorch import MAE

# Подготовка данных
nf = NeuralForecast(
    models=[
        LSTM(h=12, input_size=12, max_steps=100, random_state=42),
        GRU(h=12, input_size=12, max_steps=100, random_state=42),
        TFT(h=12, input_size=12, max_steps=100, random_state=42)
    ],
    freq='M'
)

# Обучение
print("Обучаем DL-модели (LSTM, GRU, TFT)...")
nf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')

# Прогноз
dl_forecast = nf.predict()

print("DL-прогноз готов:")
print(dl_forecast.head())

In [ ]:
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, TFT
from neuralforecast.losses.pytorch import MAE

# Подготовка данных (ts_data уже есть)
nf = NeuralForecast(
    models=[
        LSTM(h=12, input_size=12, max_steps=100),  # убрали random_state
        GRU(h=12, input_size=12, max_steps=100),   # убрали random_state
        TFT(h=12, input_size=12, max_steps=100)    # убрали random_state
    ],
    freq='M'
)

# Обучение
print("Обучаем DL-модели (LSTM, GRU, TFT)...")
nf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')

# Прогноз
dl_forecast = nf.predict()

print("DL-прогноз готов:")
print(dl_forecast.head())

In [ ]:
# Полный пайплайн анализа временных рядов (запускать одной ячейкой)
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from statsforecast import StatsForecast
from statsforecast.models import AutoARIMA, AutoETS, Theta
from mlforecast import MLForecast
from mlforecast.target_transforms import Differences
from lightgbm import LGBMRegressor
from xgboost import XGBRegressor
from catboost import CatBoostRegressor
from neuralforecast import NeuralForecast
from neuralforecast.models import LSTM, GRU, TFT
from neuralforecast.losses.pytorch import MAE

# 1. Загрузка и очистка данных
df = pd.read_csv('Space_Corrected.csv')
df = df.drop(columns=['Unnamed: 0', 'Unnamed: 0.1'])
df['Datum'] = pd.to_datetime(df['Datum'], errors='coerce')
df = df.set_index('Datum')
df['IsSuccess'] = (df['Status Mission'] == 'Success').astype(int)

# 2. Подготовка временного ряда (ежемесячная успешность)
monthly_success = df.resample('M')['IsSuccess'].mean() * 100
monthly_success = monthly_success.fillna(0)

ts_data = monthly_success.reset_index()
ts_data.columns = ['ds', 'y']
ts_data['unique_id'] = 'success_rate'

# 3. Статистические модели (StatsForecast)
models_stats = [AutoARIMA(season_length=12), AutoETS(season_length=12), Theta(season_length=12)]
sf = StatsForecast(models=models_stats, freq='M')
forecast_stats = sf.forecast(h=12, df=ts_data)

# 4. ML-модели (MLForecast)
mlf = MLForecast(
    models=[LGBMRegressor(random_state=42), XGBRegressor(random_state=42), CatBoostRegressor(random_state=42, verbose=0)],
    freq='M',
    lags=[1, 2, 3, 6, 12],
    target_transforms=[Differences([1])]
)
mlf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')
forecast_ml = mlf.predict(h=12)

# 5. DL-модели (NeuralForecast)
nf = NeuralForecast(
    models=[LSTM(h=12, input_size=12, max_steps=100), GRU(h=12, input_size=12, max_steps=100), TFT(h=12, input_size=12, max_steps=100)],
    freq='M'
)
nf.fit(ts_data, id_col='unique_id', time_col='ds', target_col='y')
forecast_dl = nf.predict()

# 6. Визуализация всех прогнозов
plt.figure(figsize=(16, 8))
plt.plot(ts_data['ds'], ts_data['y'], label='Исторические данные', color='black', alpha=0.5)

# Статистические
for model in ['AutoARIMA', 'AutoETS', 'Theta']:
    plt.plot(forecast_stats['ds'], forecast_stats[model], label=model, linestyle='--')

# ML (только первые 3 точки, чтобы не засорять график)
for model in ['LGBMRegressor', 'XGBRegressor', 'CatBoostRegressor']:
    plt.plot(forecast_ml['ds'][:3], forecast_ml[model][:3], label=model, marker='o')

# DL
for model in ['LSTM', 'GRU', 'TFT']:
    plt.plot(forecast_dl['ds'], forecast_dl[model], label=model, linewidth=2)

plt.title('Сравнение прогнозов всех 9 моделей')
plt.xlabel('Дата')
plt.ylabel('Успешность (%)')
plt.legend()
plt.grid(True)
plt.show()